In [42]:
import pandas as pd

In [52]:
dataset = "beijing-multisite-airquality" # "pv-italy" # "beijing-multisite-airquality"
data_path = f"../dvc-stages-outputs/{dataset}/raw/stations/PRSA_Data_Changping_20130301-20170228.csv"

In [53]:
df = pd.read_csv(data_path)
# if "No" in df.columns:
#     df = df.drop(columns=["No"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35064 entries, 0 to 35063
Data columns (total 18 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   No       35064 non-null  int64  
 1   year     35064 non-null  int64  
 2   month    35064 non-null  int64  
 3   day      35064 non-null  int64  
 4   hour     35064 non-null  int64  
 5   PM2.5    34290 non-null  float64
 6   PM10     34482 non-null  float64
 7   SO2      34436 non-null  float64
 8   NO2      34397 non-null  float64
 9   CO       33543 non-null  float64
 10  O3       34460 non-null  float64
 11  TEMP     35011 non-null  float64
 12  PRES     35014 non-null  float64
 13  DEWP     35011 non-null  float64
 14  RAIN     35013 non-null  float64
 15  wd       34924 non-null  str    
 16  WSPM     35021 non-null  float64
 17  station  35064 non-null  str    
dtypes: float64(11), int64(5), str(2)
memory usage: 5.2 MB


In [54]:
df

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,3.0,6.0,13.0,7.0,300.0,85.0,-2.3,1020.8,-19.7,0.0,E,0.5,Changping
1,2,2013,3,1,1,3.0,3.0,6.0,6.0,300.0,85.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Changping
2,3,2013,3,1,2,3.0,3.0,22.0,13.0,400.0,74.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Changping
3,4,2013,3,1,3,3.0,6.0,12.0,8.0,300.0,81.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Changping
4,5,2013,3,1,4,3.0,3.0,14.0,8.0,300.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Changping
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35059,35060,2017,2,28,19,28.0,47.0,4.0,14.0,300.0,NaN,11.7,1008.9,-13.3,0.0,NNE,1.3,Changping
35060,35061,2017,2,28,20,12.0,12.0,3.0,23.0,500.0,64.0,10.9,1009.0,-14.0,0.0,N,2.1,Changping
35061,35062,2017,2,28,21,7.0,23.0,5.0,17.0,500.0,68.0,9.5,1009.4,-13.0,0.0,N,1.5,Changping
35062,35063,2017,2,28,22,11.0,20.0,3.0,15.0,500.0,72.0,7.8,1009.6,-12.6,0.0,NW,1.4,Changping


In [ ]:
# show all rows having at least one NaN value
na_df = df[df.isnull().any(axis=1)]
print(f"Rows with na values: {list(na_df['No'].unique()-1)}")
na_df

In [35]:
# apply the min-max normalization to the tensor to only float columns

from sklearn.preprocessing import MinMaxScaler

float_cols = df.select_dtypes(include=['float']).columns
float_df = df[float_cols]

# impute nan columns with the mean of the column
float_df = float_df.fillna(float_df.mean())

scaler = MinMaxScaler()
normalized_tensor = scaler.fit_transform(float_df.to_numpy())
normalized_df = pd.DataFrame(normalized_tensor, columns=float_cols)
print(f"Shape of the normalized tensor: {normalized_tensor.shape}")
normalized_df

Shape of the normalized tensor: (35064, 11)


,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM
0,0.001117,0.002037,0.010902,0.017361,0.020202,0.181619,0.280977,0.661319,0.258621,0.0,0.392857
1,0.005587,0.006110,0.010902,0.017361,0.020202,0.181619,0.273997,0.664884,0.268025,0.0,0.419643
2,0.004469,0.005092,0.013837,0.027778,0.020202,0.172158,0.273997,0.670232,0.268025,0.0,0.500000
3,0.003352,0.004073,0.031447,0.031250,0.020202,0.169792,0.268761,0.688057,0.249216,0.0,0.276786
4,0.000000,0.001018,0.034382,0.034722,0.020202,0.169792,0.258290,0.700535,0.247649,0.0,0.178571
...,...,...,...,...,...,...,...,...,...,...,...
35059,0.010056,0.027495,0.013837,0.114583,0.030303,0.224193,0.511344,0.491979,0.299373,0.0,0.214286
35060,0.011173,0.035642,0.019707,0.149306,0.040404,0.191080,0.495637,0.493761,0.316614,0.0,0.080357
35061,0.014525,0.035642,0.028512,0.222222,0.060606,0.136679,0.481675,0.504456,0.344828,0.0,0.098214
35062,0.020112,0.042770,0.034382,0.295139,0.060606,0.082278,0.476440,0.508021,0.351097,0.0,0.107143


In [36]:
import numpy as np

alt_tensor = np.load(f"../dvc-stages-outputs/{dataset}/{dataset}.npz")["data"]
print(f"Shape of the alternative tensor: {alt_tensor[:, 0].dtype}")
alt_tensor[:, 0]


Shape of the alternative tensor: float64


array([[0.00111732, 0.00203666, 0.0109018 , ..., 0.25862069, 0.        ,
        0.39285714],
       [0.00558659, 0.00610998, 0.0109018 , ..., 0.26802508, 0.        ,
        0.41964286],
       [0.00446927, 0.00509165, 0.01383681, ..., 0.26802508, 0.        ,
        0.5       ],
       ...,
       [0.01452514, 0.03564155, 0.02851186, ..., 0.34482759, 0.        ,
        0.09821429],
       [0.02011173, 0.04276986, 0.03438187, ..., 0.35109718, 0.        ,
        0.10714286],
       [0.01787709, 0.02953157, 0.02851186, ..., 0.30407523, 0.        ,
        0.11607143]])

In [37]:
print(f"Shape of the normalized tensor: {normalized_tensor.dtype}")
normalized_tensor

Shape of the normalized tensor: float64


array([[0.00111732, 0.00203666, 0.0109018 , ..., 0.25862069, 0.        ,
        0.39285714],
       [0.00558659, 0.00610998, 0.0109018 , ..., 0.26802508, 0.        ,
        0.41964286],
       [0.00446927, 0.00509165, 0.01383681, ..., 0.26802508, 0.        ,
        0.5       ],
       ...,
       [0.01452514, 0.03564155, 0.02851186, ..., 0.34482759, 0.        ,
        0.09821429],
       [0.02011173, 0.04276986, 0.03438188, ..., 0.35109718, 0.        ,
        0.10714286],
       [0.01787709, 0.02953157, 0.02851186, ..., 0.30407524, 0.        ,
        0.11607143]])

In [38]:
alt_tensor[:, 0] == normalized_tensor

array([[False, False, False, ..., False,  True, False],
       [False, False, False, ..., False,  True, False],
       [False, False, False, ..., False,  True,  True],
       ...,
       [False, False, False, ..., False,  True, False],
       [False, False, False, ..., False,  True, False],
       [False, False, False, ..., False,  True, False]])

In [39]:
atol = 1
rtol = 1

for row, col in np.ndindex(alt_tensor[:, 0].shape):
    a = alt_tensor[:, 0][row, col]
    b = normalized_tensor[row, col]
    if not abs(a - b) <= atol + (rtol * b):
        print(f"Mismatch at row {row}, col {col}: a={a}, b={b}")


0.001117318
0.0011173184357541896


In [40]:
np.allclose(alt_tensor[:, 0], normalized_tensor, rtol=1, atol=1)

True

In [13]:
for i in range(alt_tensor.shape[1]):
    is_equal = np.allclose(alt_tensor[:, i], normalized_tensor, equal_nan=True, rtol=1e-05, atol=1e-08)
    print(i, ":", alt_tensor[:, i].shape, "==", normalized_tensor.shape, ":", is_equal)

0 : (35064, 11) == (35064, 11) : False
1 : (35064, 11) == (35064, 11) : False
2 : (35064, 11) == (35064, 11) : False
3 : (35064, 11) == (35064, 11) : False
4 : (35064, 11) == (35064, 11) : False
5 : (35064, 11) == (35064, 11) : False
6 : (35064, 11) == (35064, 11) : False
7 : (35064, 11) == (35064, 11) : False
8 : (35064, 11) == (35064, 11) : False
9 : (35064, 11) == (35064, 11) : False
10 : (35064, 11) == (35064, 11) : False
11 : (35064, 11) == (35064, 11) : False
